# Normalising - Cleaning up citations from raw text

Real-world legal text is messy. Citations appear in many abbreviated, fused,
or shorthand forms. The normaliser turns them into clean, canonical strings
that can be looked up in the corpus - without needing to load any law data.

In [1]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".."], check=True)
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from bundesrecht import normalise

## iVm - splitting linked references

Citations linked by `iVm` (in Verbindung mit) reference two separate legal bases.
The normaliser splits them so each can be resolved independently.

In [2]:
print(normalise("§ 312 i.V.m. § 355 BGB"))

['§ 312 BGB', '§ 355 BGB']


In [3]:
# all iVm variants are recognised
for raw in ["§ 1 iVm § 2 BGB", "§ 1 i.V.m. § 2 BGB", "§ 1 i. V. m. § 2 BGB"]:
    print(normalise(raw))

['§ 1 BGB', '§ 2 BGB']
['§ 1 BGB', '§ 2 BGB']
['§ 1 BGB', '§ 2 BGB']


## Paragraph ranges

`§§ 12-15 BGB` is shorthand for four separate references.
The normaliser expands these into individual canonical strings.

In [4]:
print(normalise("§§ 12-15 BGB"))

['§ 12 BGB', '§ 13 BGB', '§ 14 BGB', '§ 15 BGB']


In [5]:
# bis ranges work the same way
print(normalise("§§ 12 bis 15 BGB"))

['§ 12 BGB', '§ 13 BGB', '§ 14 BGB', '§ 15 BGB']


## Multi-target citations

A single citation can reference multiple sub-targets within the same paragraph.
The normaliser expands each into its own canonical string.

In [6]:
print(normalise("§ 2 Abs. 1 Nr. 1, Nr. 7, Abs. 2 UrhG"))

['§ 2 Abs. 1 Nr. 1 UrhG', '§ 2 Abs. 1 Nr. 7 UrhG', '§ 2 Abs. 2 UrhG']


In [7]:
# Abs. 1 und 2 also expands
print(normalise("§ 433 Abs. 1 und 2 BGB"))

['§ 433 Abs. 1 BGB', '§ 433 Abs. 2 BGB']


## Mixed multi-law citations

Citations like `§§ 46 Abs. 2 ArbGG, 91 Abs. 1 ZPO` reference different laws
in a single string. The normaliser splits and assigns the correct law to each.

In [8]:
print(normalise("§§ 46 Abs. 2 ArbGG, 91 Abs. 1 ZPO"))

['§ 46 Abs. 2 ArbGG', '§ 91 Abs. 1 ZPO']


In [9]:
# shared law at the end applies to all paragraphs
print(normalise("§§ 137 S. 2, 398, 903 BGB"))

['§ 137 Satz 2 BGB', '§ 398 BGB', '§ 903 BGB']


## Shorthand expansion

The normaliser expands common shorthand forms that appear in practice.

In [10]:
# S. expands to Satz
print(normalise("§ 433 Abs. 1 S. 2 BGB"))

['§ 433 Abs. 1 Satz 2 BGB']


In [11]:
# Roman numeral Absatz shorthand: § 62 I 2 -> § 62 Abs. 1 Satz 2
print(normalise("§ 62 I 2 AufenthG"))

['§ 62 Abs. 1 Satz 2 AufenthG']


In [12]:
# Ziffer is a synonym for Nr.
print(normalise("§ 23 Abs. 2 Ziffer 2 SGB VIII"))

['§ 23 Abs. 2 Nr. 2 SGB 8']


In [13]:
# no-space between § and paragraph number
print(normalise("§312 BGB"))

['§ 312 BGB']


## Continuation markers - f. and ff.

`f.` (und folgende) always expands to exactly 2 paragraphs.
`ff.` (und fortfolgende) is preserved by default since the intended
range is ambiguous - pass `ff_expansion` to expand to a specific count.

In [14]:
# f. always expands to 2
print(normalise("§ 312 f. BGB"))

['§ 312 BGB', '§ 313 BGB']


In [15]:
# ff. preserved by default
print(normalise("§ 312 ff. BGB"))

['§ 312 ff. BGB']


In [16]:
# ff. expanded when ff_expansion is given
print(normalise("§ 312 ff. BGB", ff_expansion=3))

['§ 312 BGB', '§ 313 BGB', '§ 314 BGB']


In [17]:
print(normalise("§ 312 ff. BGB", ff_expansion=5))

['§ 312 BGB', '§ 313 BGB', '§ 314 BGB', '§ 315 BGB', '§ 316 BGB']


## Case study - cleaning a batch of raw citations

In practice you might extract citations from a court decision and need to
normalise them all before resolving. Here is a typical batch.

In [18]:
raw_citations = [
    "§ 242 BGB",
    "§§ 433, 434 Abs. 1 BGB",
    "§ 823 Abs. 1 i.V.m. § 1004 BGB",
    "§ 62 I 2 AufenthG",
    "§312 StGB",
    "§ 2 Abs. 1 Nr. 1 und 2 UrhG",
]

canonical = []
for raw in raw_citations:
    canonical.extend(normalise(raw))

for c in canonical:
    print(c)

§ 242 BGB
§ 433 BGB
§ 434 Abs. 1 BGB
§ 823 Abs. 1 BGB
§ 1004 BGB
§ 62 Abs. 1 Satz 2 AufenthG
§ 312 StGB
§ 2 Abs. 1 Nr. 1 UrhG
§ 2 Abs. 1 Nr. 2 UrhG
